<a href="https://colab.research.google.com/github/Tanmay-Somani/100daysof-code/blob/main/Main_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import re
import logging
from datetime import datetime
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import random
from torchsummary import summary
from skimage.metrics import peak_signal_noise_ratio as psnr_metric, structural_similarity as ssim_metric

In [ ]:
SEQ_LEN = 7
PRED_STEPS = 1
FULL_IMG_SIZE = 1500
IMG_SIZE=256
BATCH_SIZE = 1
EPOCHS = 40 # Increased epochs to allow early stopping to kick in
LR = 5e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED=42

TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15 # Remaining for test (0.15)

In [ ]:
BASE_DIR = "/content/project"

os.makedirs(BASE_DIR, exist_ok=True)

DATA_ROOT = f"{BASE_DIR}/data"
LOG_PATH = f"{BASE_DIR}/logs"
LOG_FILE=LOG_PATH+ "/train_cropped.log"
MODEL_PATH = f"{BASE_DIR}/models"
CHKPOINT_DIR = f"{MODEL_PATH}/checkpoints"
IMAGES_DIR = f"{BASE_DIR}/images" # New: Define images directory

for p in [DATA_ROOT, LOG_PATH, MODEL_PATH, CHKPOINT_DIR, IMAGES_DIR]: # New: Add IMAGES_DIR
    os.makedirs(p, exist_ok=True)

In [ ]:
with open(LOG_PATH+'/runno.txt','r') as  R:
    runno=int(R.read())
with open(LOG_PATH+'/runno.txt','w') as  W:
    W.write(str(int(runno+1)))
logging.basicConfig(level=logging.INFO,format="%(asctime)s | %(levelname)s | %(message)s",
                    handlers=[logging.FileHandler(LOG_FILE),
                              logging.StreamHandler()]
)

In [ ]:

def set_seed(seed):
    """
    Sets the seeds for `random`, `numpy`, and `torch` libraries to ensure reproducibility.

    Args:
        seed (int): The seed value to be used.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [ ]:




def parse_datetime(path):
    """
    Parses a datetime string from a file path.

    The file path is expected to be in the format "MMWYY_HHMM" or "MWWYY_HHMM", where:
        - MM: Month as a two-digit number (01-12)
        - W: Weekday (JAN-FEB etc.)
        - YY: Year as a four-digit number
        - HH: Hour in 24-hour format (00-23)
        - MM: Minute (00-59)

    Args:
        path (str): The file path to parse the datetime from.

    Returns:
        datetime: A datetime object representing the parsed date and time.

    Raises:
        ValueError: If the file path does not match the expected format.
    """
    name = os.path.basename(path)
    pattern = r"(\d{2})(\w{3})(\d{4})_(\d{4})"
    match = re.search(pattern, name)
    if not match:
        raise ValueError(f"Bad filename: {name}")
    month_map = {
        'JAN': '01','FEB': '02','MAR': '03','APR': '04','MAY': '05','JUN': '06',
        'JUL': '07','AUG': '08','SEP': '09','OCT': '10','NOV': '11','DEC': '12'
    }
    day=match.group(1)
    month = match.group(2)
    year=match.group(3)
    time=match.group(4)
    for key, value in month_map.items():
        if key in month:
            month = value
            break
    return datetime.strptime(f"{day}{month}{year}.{time}", "%d%m%Y.%H%M")


In [ ]:

def mse(p,t):
    """
    Computes the mean squared error (MSE) between two tensors.

    Args:
        p (torch.tensor): The predicted tensor.
        t (torch.tensor): The target tensor.

    Returns:
        torch.tensor: The mean squared error value.
    """
    return torch.mean((p-t)**2)

def rmse(p,t):
    """
    Computes the root mean squared error (RMSE) between two tensors.

    Args:
        p (torch.tensor): The predicted tensor.
        t (torch.tensor): The target tensor.

    Returns:
        torch.tensor: The root mean squared error value.
    """
    return torch.sqrt(mse(p,t)+1e-8)

def mae(p,t):
    """
    Computes the mean absolute error (MAE) between two tensors.

    Args:
        p (torch.tensor): The predicted tensor.
        t (torch.tensor): The target tensor.

    Returns:
        torch.tensor: The mean absolute error value.
    """
    return torch.mean(torch.abs(p-t))

def encode_time(dt):
    """
    Encodes a datetime object into sinusoidal representation.

    The encoding is based on the hour of the day, with values scaled to be between -1 and 1.

    Args:
        dt (datetime): The datetime object to encode.

    Returns:
        tuple: A pair of numpy arrays representing the sine and cosine components of the encoded time.
    """
    hour = dt.hour + dt.minute / 60.0
    sin = np.sin(2 * np.pi * hour / 24)
    cos = np.cos(2 * np.pi * hour / 24)
    return sin, cos

def loss_fn(pred, target):
    mse = ((pred-target)**2).mean()
    mae = torch.abs(pred-target).mean()
    return mse + 0.1*mae, mse.item(), mae.item()

def calculate_psnr(p, t):
    """Calculates PSNR between two tensors, handling batch, sequence, and channel dimensions."""
    # Ensure tensors are on CPU and converted to numpy, then squeeze redundant dimensions
    p_np = p.detach().cpu().numpy().squeeze()
    t_np = t.detach().cpu().numpy().squeeze()

    if p_np.ndim == 2: # Single image (H, W)
        return psnr_metric(t_np, p_np, data_range=1.0)
    elif p_np.ndim == 3: # Batch of images (B, H, W)
        batch_psnr = [psnr_metric(t_np[i], p_np[i], data_range=1.0) for i in range(p_np.shape[0])]
        return np.mean(batch_psnr)
    else:
        raise ValueError(f"Unexpected number of dimensions for PSNR calculation: {p_np.ndim}")

def calculate_ssim(p, t):
    """Calculates SSIM between two tensors, handling batch, sequence, and channel dimensions."""
    # Ensure tensors are on CPU and converted to numpy, then squeeze redundant dimensions
    p_np = p.detach().cpu().numpy().squeeze()
    t_np = t.detach().cpu().numpy().squeeze()

    if p_np.ndim == 2: # Single image (H, W)
        return ssim_metric(t_np, p_np, data_range=1.0)
    elif p_np.ndim == 3: # Batch of images (B, H, W)
        batch_ssim = [ssim_metric(t_np[i], p_np[i], data_range=1.0) for i in range(p_np.shape[0])]
        return np.mean(batch_ssim)
    else:
        raise ValueError(f"Unexpected number of dimensions for SSIM calculation: {p_np.ndim}")

In [ ]:
class SpatioTemporalDataset(Dataset):
    """
    Initializes the SpatioTemporalDataset class.

    Args:
        image_paths (list): A list of image file paths for the dataset.

    Note:
        This function takes pre-sorted image paths and creates a list of tuples containing input and target sequences.

    Attributes:
        samples (list[tuple]): A list of tuples, where each tuple contains an input sequence and a corresponding target sequence.
    """
    def __init__(self, image_paths):
        self.image_paths = image_paths
        self.samples = []
        total_len = SEQ_LEN + PRED_STEPS

        for i in range(len(self.image_paths) - total_len):
            window = self.image_paths[i:i+total_len]
            inp = window[:SEQ_LEN]
            tgt = window[SEQ_LEN:]
            self.samples.append((inp, tgt))

        logging.info(f"Total valid samples for this split: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def load_img(self, path):
        """
        Load an image from a file.

        Args:
            path (str): The path to the image file.

        Returns:
            np.ndarray: The loaded image.
        """
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(
            img,
            (IMG_SIZE, IMG_SIZE),
            interpolation=cv2.INTER_AREA
        )
        img = img / 255.0
        return img

    def __getitem__(self, idx):
        seq = []
        targets = []
        inp_paths, tgt_paths = self.samples[idx]

        for p in inp_paths:
            img = self.load_img(p)
            dt = parse_datetime(p)
            sin, cos = encode_time(dt)
            time_map = np.ones_like(img)
            stacked = np.stack(
                [
                    img,
                    sin * time_map,
                    cos * time_map
                ],
                axis=0
            )
            seq.append(stacked)
        seq = np.stack(seq, axis=0)

        for p in tgt_paths:
            img = self.load_img(p)
            targets.append(np.expand_dims(img, axis=0))
        targets = np.stack(targets, axis=0)
        return torch.tensor(seq, dtype=torch.float32), torch.tensor(targets, dtype=torch.float32)

In [ ]:
class ConvLSTMCell(nn.Module):
    """
    A 2D Convolutional LSTM Cell.

    Attributes:
        hidden_dim (int): The number of output features in the cell.
        conv (nn.Conv2d): The convolutional layer that combines input and hidden state.

    Methods:
        forward(x, h, c): The forward pass through the cell.
    """
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv2d(
        in_dim + hidden_dim,
        4 * hidden_dim,
        3,
        padding=1
        )

    def forward(self, x, h, c):
        """
        The forward pass through the cell.

        Args:
            x (torch.Tensor): The input tensor.
            h (torch.Tensor): The current hidden state.
            c (torch.Tensor): The current cell state.

        Returns:
            tuple: A tuple containing the updated hidden and cell states.
        """

        combined = torch.cat([x, h], dim=1)
        out = self.conv(combined)
        i, f, o, g = torch.chunk(out, 4, dim=1)
        i=torch.sigmoid(i)
        f=torch.sigmoid(f)
        o=torch.sigmoid(o)
        g=torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        return h, c

# =========================
# MODEL (Encoder-Decoder ConvLSTM)
# =========================
class ConvLSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Define hidden dimensions for each layer of the encoder and decoder
        # Increased layers and channels for more capacity
        self.enc_hidden_dims = [32, 64, 32] # Example: 3 encoder layers with specified hidden channels
        self.dec_hidden_dims = [32, 64, 32] # Example: 3 decoder layers, can mirror encoder or be different

        self.encoder_cells = nn.ModuleList()
        # First encoder cell takes input_channels (3 for image + time) as its input_dim
        self.encoder_cells.append(ConvLSTMCell(3, self.enc_hidden_dims[0]))
        # Subsequent encoder cells take hidden_dim of previous cell as their input_dim
        for i in range(1, len(self.enc_hidden_dims)):
            self.encoder_cells.append(ConvLSTMCell(self.enc_hidden_dims[i-1], self.enc_hidden_dims[i]))

        self.decoder_cells = nn.ModuleList()
        # First decoder cell's input_dim usually matches the last encoder cell's hidden_dim to pass context
        self.decoder_cells.append(ConvLSTMCell(self.enc_hidden_dims[-1], self.dec_hidden_dims[0]))
        # Subsequent decoder cells take hidden_dim of previous cell as their input_dim
        for i in range(1, len(self.dec_hidden_dims)):
            self.decoder_cells.append(ConvLSTMCell(self.dec_hidden_dims[i-1], self.dec_hidden_dims[i]))

        # Output head: converts the last decoder layer's hidden state to the desired output prediction steps
        self.head = nn.Conv2d(self.dec_hidden_dims[-1], PRED_STEPS, kernel_size=1)

    def forward(self, x):
        B, T_enc, C, H, W = x.shape # T_enc is SEQ_LEN, C is 3 (image + time features)

        # Initialize encoder hidden and cell states for all layers
        encoder_h_c_states = []
        for hidden_dim in self.enc_hidden_dims:
            encoder_h_c_states.append((
                torch.zeros(B, hidden_dim, H, W, device=x.device),
                torch.zeros(B, hidden_dim, H, W, device=x.device)
            ))

        # Encoder pass: Process each time step of the input sequence through all layers
        for t in range(T_enc):
            current_layer_input = x[:, t] # Input for the first encoder layer at current time step
            for i, cell in enumerate(self.encoder_cells):
                h, c = cell(current_layer_input, encoder_h_c_states[i][0], encoder_h_c_states[i][1])
                encoder_h_c_states[i] = (h, c)
                current_layer_input = h # Output (h) of current layer becomes input for next layer at same time step

        # Initialize decoder states with the final states from the encoder
        # Use a deep copy to ensure decoder states are independent of encoder states if needed for future modifications
        decoder_h_c_states = list(encoder_h_c_states)

        # Decoder pass: Generate PRED_STEPS future predictions
        outputs = []
        # The input to the first decoder cell for each prediction step is typically a context vector or a zero tensor.
        # Here, we use a zero tensor matching the input size of the first decoder cell.
        decoder_input_for_first_layer = torch.zeros(B, self.enc_hidden_dims[-1], H, W, device=x.device)

        for t_pred in range(PRED_STEPS):
            current_layer_input = decoder_input_for_first_layer # This input is fed to the first decoder layer at each prediction step

            for i, cell in enumerate(self.decoder_cells):
                h, c = cell(current_layer_input, decoder_h_c_states[i][0], decoder_h_c_states[i][1])
                decoder_h_c_states[i] = (h, c)
                current_layer_input = h # Output (h) of current layer becomes input for next layer at same time step

            # After processing all decoder layers for this prediction time step, apply the head
            outputs.append(self.head(decoder_h_c_states[-1][0])) # Apply head to the last decoder layer's hidden state

        # Stack all predicted time steps and add a channel dimension (C=1)
        return torch.stack(outputs, dim=1).unsqueeze(2) # (B, PRED_STEPS, C=1, H, W)

def save_predictions(pred, target, epoch, split_name="train"):
    """
    Save predictions to files.

    Args:
        pred (torch.Tensor): The predicted tensor.
        target (torch.Tensor): The target tensor.
        epoch (int): The current epoch.
        split_name (str): Name of the dataset split (e.g., 'train', 'val', 'test').
    """
    pred = pred.detach().cpu().numpy()
    target = target.detach().cpu().numpy()

    pred_dir = os.path.join(BASE_DIR, f"images/preds/preds_main_cropped/{split_name}")
    os.makedirs(pred_dir, exist_ok=True)

    for i in range(min(2, pred.shape[0])):
        for t in range(PRED_STEPS):
            plt.imsave(os.path.join(pred_dir, f"epoch{epoch}_sample{i}_t{t}.png"), pred[i, t, 0], cmap='gray')

def save_ckpt(model, opt, epoch, is_best=False):
    """
    Save a checkpoint to file.

    Args:
        model (nn.Module): The model to be saved.
        opt (torch.optim.Optimizer): The optimizer to be saved.
        epoch (int): The current epoch.
        is_best (bool): If True, saves as the best model.
    """
    os.makedirs(CHKPOINT_DIR, exist_ok=True)
    path = os.path.join(CHKPOINT_DIR, f"epoch_{epoch}.pth")
    torch.save({
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "epoch": epoch
    }, path)
    logging.info(f"Checkpoint saved: {path}")

    if is_best:
        best_path = os.path.join(CHKPOINT_DIR, "best_model.pth")
        torch.save(model.state_dict(), best_path)
        logging.info(f"Best model saved: {best_path}")

In [ ]:
def train_epoch(model, train_loader, opt, epoch, hist, DEVICE, loss_fn, mae, mse, rmse, calculate_psnr, calculate_ssim):
    model.train()
    train_totals = {k: 0 for k in ["loss", "mae", "mse", "rmse", "psnr", "ssim"]}
    train_cnt = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]")

    for x, y in pbar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        pred = model(x)
        loss, mse_val, mae_val = loss_fn(pred, y)
        loss.backward()
        opt.step()

        with torch.no_grad():
            train_totals["loss"] += loss.item()
            train_totals["mae"] += mae(pred, y).item()
            train_totals["mse"] += mse(pred, y).item()
            train_totals["rmse"] += rmse(pred, y).item()
            train_totals["psnr"] += calculate_psnr(pred, y)
            train_totals["ssim"] += calculate_ssim(pred, y)
            train_cnt += 1

        pbar.set_postfix(loss=loss.item())

    for k in train_totals: hist[f"train_{k}"].append(train_totals[k] / train_cnt if train_cnt > 0 else 0)
    logging.info(f"Epoch {epoch+1} [Train] Avg Loss: {hist['train_loss'][-1]:.4f}, MAE: {hist['train_mae'][-1]:.4f}, PSNR: {hist['train_psnr'][-1]:.4f}, SSIM: {hist['train_ssim'][-1]:.4f}")

def validate_epoch(model, val_loader, epoch, hist, DEVICE, loss_fn, mae, mse, rmse, calculate_psnr, calculate_ssim):
    model.eval()
    val_totals = {k: 0 for k in ["loss", "mae", "mse", "rmse", "psnr", "ssim"]}
    val_cnt = 0
    pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]")

    with torch.no_grad():
        for x, y in pbar_val:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            loss, mse_val, mae_val = loss_fn(pred, y)

            val_totals["loss"] += loss.item()
            val_totals["mae"] += mae(pred, y).item()
            val_totals["mse"] += mse(pred, y).item()
            val_totals["rmse"] += rmse(pred, y).item()
            val_totals["psnr"] += calculate_psnr(pred, y)
            val_totals["ssim"] += calculate_ssim(pred, y)
            val_cnt += 1

            pbar_val.set_postfix(loss=loss.item())

    for k in val_totals: hist[f"val_{k}"].append(val_totals[k] / val_cnt if val_cnt > 0 else 0)
    logging.info(f"Epoch {epoch+1} [Val] Avg Loss: {hist['val_loss'][-1]:.4f}, MAE: {hist['val_mae'][-1]:.4f}, PSNR: {hist['val_psnr'][-1]:.4f}, SSIM: {hist['val_ssim'][-1]:.4f}")
    return hist['val_loss'][-1]

def test_model(model, test_loader, DEVICE, loss_fn, mae, mse, rmse, calculate_psnr, calculate_ssim):
    model.eval()
    test_totals = {k: 0 for k in ["loss", "mae", "mse", "rmse", "psnr", "ssim"]}
    test_cnt = 0
    pbar_test = tqdm(test_loader, desc="[Test]")

    if os.path.exists(os.path.join(CHKPOINT_DIR, "best_model.pth")):
        model.load_state_dict(torch.load(os.path.join(CHKPOINT_DIR, "best_model.pth")))
        logging.info("Loaded best model for testing.")

    with torch.no_grad():
        for x, y in pbar_test:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x)
            loss, mse_val, mae_val = loss_fn(pred, y)

            test_totals["loss"] += loss.item()
            test_totals["mae"] += mae(pred, y).item()
            test_totals[

In [ ]:
if __name__ == "__main__":
    logging.info('----------------------------------------------------------------')
    logging.info("Starting Project...")
    logging.info(f"Run No. {runno}")
    logging.info("Configurations: ")
    logging.info("Sequence length of images taken "+ str(SEQ_LEN))
    logging.info("Current Batch Size taken " + str(BATCH_SIZE))
    logging.info("Number of Epochs in the run "+str(EPOCHS))
    logging.info("Current training image resolution " + str(IMG_SIZE))
    logging.info("Learning Rate "+str(LR))
    logging.info(f"DEVICE {DEVICE}")
    set_seed(SEED)

    # Explicitly define split constants here to ensure they are available
    # if the previous configuration cell was not executed or updated.
    TRAIN_SPLIT = 0.7
    VAL_SPLIT = 0.15
    TEST_SPLIT = 0.15 # Remaining for test (0.15)

    # Set this to a number (e.g., 200) to limit the total images for debugging,
    # or None to use all available images.
    DEBUG_MAX_IMAGES = 60 # Changed from 35 to ensure validation and test sets are not empty

    try:
        run_experiment(max_images=DEBUG_MAX_IMAGES)
    except Exception as e:
        logging.exception(f"Fatal error: {e}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1          [1, 64, 256, 256]          11,008
      ConvLSTMCell-2  [[-1, 16, 256, 256], [-1, 16, 256, 256]]               0
            Conv2d-3          [1, 64, 256, 256]          18,496
      ConvLSTMCell-4  [[-1, 16, 256, 256], [-1, 16, 256, 256]]               0
            Conv2d-5          [1, 64, 256, 256]          11,008
      ConvLSTMCell-6  [[-1, 16, 256, 256], [-1, 16, 256, 256]]               0
            Conv2d-7          [1, 64, 256, 256]          18,496
      ConvLSTMCell-8  [[-1, 16, 256, 256], [-1, 16, 256, 256]]               0
            Conv2d-9          [1, 64, 256, 256]          11,008
     ConvLSTMCell-10  [[-1, 16, 256, 256], [-1, 16, 256, 256]]               0
           Conv2d-11          [1, 64, 256, 256]          18,496
     ConvLSTMCell-12  [[-1, 16, 256, 256], [-1, 16, 256, 256]]               0
           Co

[Test]: 100%|██████████| 1/1 [00:01<00:00,  1.78s/it, loss=0.009]
